In [1]:
import os, sys, joblib
import numpy as np

sys.path.append("YOUR-PATH/actionable-hypotension/")

from evaluation.utils.evaluation_functions import load_and_prepare_data_xgb, load_and_prepare_data_baseline
from evaluation.utils.calculate_tables import (                            
    build_all_tables_for_model
)

CALIBRATED_MODELS_DIR = "../models_given/calibrated"
FIGURES_DIR = "figures_new"

/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class_name = "baseline" # <-- select model

### Help Functions

In [3]:
# Load validation and test splits from the DB (your function)
x_val, y_val, x_test, y_test, feat_cols = None, None, None, None, None
if class_name == "baseline":
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_baseline()
else:
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_xgb(
        table_name=f"merged_{class_name}_features"

    )
    
y_val  = np.asarray(y_val).ravel()
y_test = np.asarray(y_test).ravel()

### Load model

In [4]:
# Load calibrated model
if class_name == "baseline":
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr.pkl"))
else:
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, f"xgb_{class_name}.pkl"))

# Use positive-class probabilities
y_val_pred  = model.predict_proba(x_val)
y_test_pred = model.predict_proba(x_test)
y_val_scores  = y_val_pred[:, 1] if y_val_pred.ndim == 2 else y_val_pred
y_test_scores = y_test_pred[:, 1] if y_test_pred.ndim == 2 else y_test_pred

/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Calculate Metric Tables

In [5]:
tables = build_all_tables_for_model(
    class_name=class_name,
    y_val=y_val,          y_val_pred=y_val_scores,
    y_test=y_test,        y_test_pred=y_test_scores,
    targets=(0.70, 0.75, 0.80, 0.85, 0.90),
    main_target=0.80, 
    n_boot=1000, alpha=0.95, seed=42,
    figures_dir=FIGURES_DIR,
    strategy="quantile_pos",
    rule=">=",
)

In [6]:
tables

{'table4_val': None,
 'table4_test':   ThresholdType  Target      Sens      Spec       PPV       NPV        F1  \
 0          0.70    0.70  0.772152  0.467783  0.002873  0.999034  0.005724   
 1          0.75    0.75  0.772152  0.467783  0.002873  0.999034  0.005724   
 2          0.80    0.80  0.830018  0.366165  0.002594  0.999079  0.005171   
 3          0.85    0.85  0.854430  0.320645  0.002491  0.999099  0.004968   
 4          0.90    0.90  0.931736  0.150685  0.002174  0.999101  0.004337   
 5        Youden     NaN  0.575949  0.718563  0.004047  0.998829  0.008038   
 
         Thr  Sens_selected  
 0  0.001320       0.772152  
 1  0.001320       0.772152  
 2  0.001081       0.830018  
 3  0.000947       0.854430  
 4  0.000853       0.931736  
 5  0.002174            NaN  ,
 'table5_val': None,
 'table5_test':   Metric   Value  CI_low  CI_high
 0     TP    1836    1752     1924
 1     FP  706040  705171   706923
 2     TN  407878  406971   408742
 3     FN     376     337    